In [ ]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import SimpleITK as sitk
from radiomics import featureextractor
import pickle
import os

In [ ]:
#LOAD All MODELS AND FEATURE INDEXES
model_path = "Path to models"
Dmodel = load_model(model_path+"/Best DenseNet Model.keras")
FeatDModel = Model(inputs=Dmodel.inputs, outputs=Dmodel.get_layer('global_max_pooling2d').output)
Xmodel = load_model(model_path+"/Best Xception model.keras")
FeatXModel = Model(inputs=Xmodel.inputs, outputs=Xmodel.get_layer('global_max_pooling2d_1').output)
Rmodel = load_model(model_path+"/Best ResNet101 model.keras")
FeatRModel = Model(inputs=Rmodel.inputs, outputs=Rmodel.get_layer('global_max_pooling2d').output)
selectedndx = np.load("selectedfeatndx.npy")
D_ndx = []
X_ndx = []
R_ndx = []
for i in range(2150):
    if selectedndx[i]<1024:
        D_ndx.append(selectedndx[i])
    if selectedndx[i]>=1024 and selectedndx[i]<3072:
        X_ndx.append(selectedndx[i]-1024)
    if selectedndx[i]>=3072:
        R_ndx.append(selectedndx[i]-3072)
Ra_ndx = np.load("Path to Radsigfeats.npy")
Ra_min = np.load("Path to Ra_scaler_min.npy")
Ra_max = np.load("Path to Ra_scaler_max.npy")
clindf = pd.read_csv("Path to clinicaldata.csv")
D_clf = pickle.load(open("Path to D_MLP.sav", 'rb'))
X_clf = pickle.load(open("Path to X_MLP.sav", 'rb'))
R_clf = pickle.load(open("Path to R_MLP.sav", 'rb'))
Ra_clf = pickle.load(open("Path to Ra_MLP.sav", 'rb'))
Cl_clf = pickle.load(open("Path to Cl_MLP.sav", 'rb'))

In [ ]:
#DEFINE ALL FUNCTIONS

def extract_dl_features(impath):
    my_image = load_img(impath, target_size=(224, 224))
    my_image = img_to_array(my_image)
    my_image = my_image.reshape((1, 224, 224, 3))
    my_image = my_image/255.
    Dfeatures = FeatDModel.predict(my_image)[0,D_ndx]
    Xfeatures = FeatXModel.predict(my_image)[0,X_ndx]
    Rfeatures = FeatRModel.predict(my_image)[0,R_ndx]
    return Dfeatures, Xfeatures, Rfeatures

def extract_ra_features(itkimg):
    mask_array = np.ones(itkimg.GetSize()[::-1], dtype=np.int32)
    mask = sitk.GetImageFromArray(mask_array)
    mask.SetSpacing(itkimg.GetSpacing())
    mask.SetOrigin(itkimg.GetOrigin())
    mask.SetDirection(itkimg.GetDirection())
    extractor = featureextractor.RadiomicsFeatureExtractor()
    result = extractor.execute(itkimg, mask, label=1)
    redf = pd.DataFrame([result])
    rearr = redf.iloc[0].to_numpy()
    drop_cols = [0,1,2,3,4,5,6,7,8,9,10,14,15,16,17,18,19,20,21]
    rearr = np.delete(rearr,drop_cols)
    rearr = (rearr-Ra_min)/(Ra_max-Ra_min)
    rearr = rearr[Ra_ndx]
    return rearr

def extract_cl_features(imgid):
    cldf = clindf[clindf['alias']==imgid]
    cld = cldf.iloc[0,1:].to_numpy()
    return cld

def extract_all_features(impath, imgid):
    d,x,r = extract_dl_features(impath)
    rimg = sitk.ReadImage(impath)
    ra = extract_ra_features(rimg)
    cl = extract_cl_features(imgid)
    d=np.reshape(d,(1,d.shape[0]))
    x=np.reshape(x,(1,x.shape[0]))
    r=np.reshape(r,(1,r.shape[0]))
    ra=np.reshape(ra,(1,ra.shape[0]))
    cl=np.reshape(cl,(1,cl.shape[0]))
    return d,x,r,ra,cl

def get_ensemble_prob(d,x,r,ra,cl):
    d_prob = D_clf.predict_proba(d)[0][1]
    x_prob = X_clf.predict_proba(x)[0][1]
    r_prob = R_clf.predict_proba(r)[0][1]
    ra_prob = Ra_clf.predict_proba(ra)[0][1]
    cl_prob = Cl_clf.predict_proba(cl)[0][1]
    d_o = get_odds(d_prob)
    x_o = get_odds(x_prob)
    r_o = get_odds(r_prob)
    ra_o = get_odds(ra_prob)
    cl_o = get_odds(cl_prob)
    only_xray_o = d_o*x_o*r_o*ra_o*cl_o
    with_clin_o = d_o*x_o*r_o*ra_o
    only_xray_p = get_prob(only_xray_o)
    with_clin_p = get_prob(with_clin_o)
    return only_xray_p, with_clin_p

def get_odds(pr):
    if pr<0.0001:
        pr=0.0001
    if pr>0.9999:
        pr=0.9999
    return pr/(1-pr)

def get_prob(od):
    return od/(1+od)

In [ ]:
#PREDICT A SINGLE IMAGE
impath = "Path to image.jpg"
df,xf,rf,raf,clf = extract_all_features(impath, imid)
x_only, x_and_c = get_ensemble_prob(df,xf,rf,raf,clf)
print(x_only, x_and_c)

In [ ]:
#PREDICT ALL IMAGES IN A FOLDER
fpath = "Path to image folder/"
fl = os.listdir(fpath)
imlist = [f[:-4] for f in fl]
xlist=[]
xclist=[]
for i in range(len(imlist)):
    df,xf,rf,raf,clf = extract_all_features(fpath+fl[i],imlist[i])
    x_only, x_and_c = get_ensemble_prob(df,xf,rf,raf,clf)
    xlist.append(x_only)
    xclist.append(x_and_c)
    print(str(i+1), fpath+fl[i], x_only, x_and_c)